# 02 — Machine-learning problem definition

The goal is to predict, **before departure**, whether a future flight will arrive at least 15 minutes late. This notebook defines the target, eligible population, feature boundary, and chronological evaluation splits. It does not train a model.

## 1. Prediction contract

- **Unit of prediction:** one scheduled flight.
- **Prediction moment:** before the flight departs.
- **Target:** `label = 1` when `ArrDelayMinutes >= 15`; otherwise `label = 0`.
- **Eligible population:** completed, non-diverted flights with a known arrival-delay outcome.
- **Excluded outcomes:** cancelled and diverted flights. They represent different operational problems and do not have a comparable final-arrival target.

The 15-minute threshold follows the standard delayed-flight definition used throughout this project.

## 2. Features available before departure versus leakage

A feature is valid only if it would be known at the prediction moment. Using fields produced after departure would let the model see part of the answer.

### Valid baseline pre-flight features

- Calendar: `Month`, `DayofMonth`, `DayOfWeek` and date-derived seasonality.
- Airline and service: `Reporting_Airline`, `Flight_Number_Reporting_Airline`.
- Network: `Origin`, `Dest`, origin/destination state or market identifiers.
- Schedule: `CRSDepTime`, `CRSArrTime`, `DepTimeBlk`, `ArrTimeBlk`, `CRSElapsedTime`.
- Route length: `Distance`, `DistanceGroup`.

### Conditionally available; excluded from the baseline

`Tail_Number` may be assigned before departure, but aircraft assignments can change and may not be reliably available at scoring time. Keep it out of the initial feature set unless the production data contract guarantees availability.

### Redundant identifiers

DOT, airport-sequence, and related numeric identifiers often repeat information already represented by airline and airport codes. They can be evaluated later but are not necessary for the first model definition.

### Leakage and post-flight fields

Do not use `DepTime`, departure-delay fields, `TaxiOut`, `WheelsOff`, `WheelsOn`, `TaxiIn`, `ArrTime`, arrival-delay fields, `ActualElapsedTime`, `AirTime`, `Cancelled`, `CancellationCode`, `Diverted`, diversion details, or any delay-cause field. These values are observed during or after operations and would reveal the outcome.

## 3. Load the source data

The notebook inherits Spark master and worker settings from the project runtime. It does not install PySpark or configure a cluster.

In [1]:
from pathlib import Path

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F

def find_project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/raw/csv").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate data/raw/csv/. Run this notebook inside the Airline project.")

PROJECT_ROOT = find_project_root()
DATA_REL = Path("data/raw/csv")
DATA_DIR = PROJECT_ROOT / DATA_REL

spark = SparkSession.builder.appName("airline-02-ml-problem-definition").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

flights = (
    spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "*.csv"))
    .withColumn("flight_date", F.to_date("FlightDate"))
)
print("Spark session ready.")
print(f"Raw input: {DATA_REL.as_posix()}/")

Spark session ready.
Raw input: data/raw/csv/


## 4. Build the eligible labeled population

Eligibility is applied before assigning the label. This prevents a cancelled or diverted flight with no arrival outcome from being incorrectly treated as on time.

In [2]:
baseline_features = [
    "flight_date", "Month", "DayofMonth", "DayOfWeek",
    "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "OriginState", "OriginCityMarketID",
    "Dest", "DestState", "DestCityMarketID",
    "CRSDepTime", "CRSArrTime", "DepTimeBlk", "ArrTimeBlk",
    "CRSElapsedTime", "Distance", "DistanceGroup",
]

eligible = (
    flights
    .filter(
        (F.col("Cancelled") == 0)
        & (F.col("Diverted") == 0)
        & F.col("ArrDelayMinutes").isNotNull()
        & F.col("flight_date").isNotNull()
    )
    .withColumn("label", (F.col("ArrDelayMinutes") >= 15).cast("int"))
    .select(*baseline_features, "label")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

eligible.agg(
    F.count("*").alias("eligible_rows"),
    F.sum("label").alias("delayed_rows"),
    F.round(F.avg("label") * 100, 3).alias("delayed_proportion_pct"),
).show(truncate=False)
eligible.printSchema()

+-------------+------------+----------------------+
|eligible_rows|delayed_rows|delayed_proportion_pct|
+-------------+------------+----------------------+
|13708650     |2836665     |20.693                |
+-------------+------------+----------------------+

root
 |-- flight_date: date (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- DestCityMarketID: integer (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- DepTimeBlk: string (nullable = true)
 |-- ArrTimeBlk: string (nullable = true)
 |-- CRSElapsedTime: double (

The resulting frame contains only candidate pre-flight fields plus the label. `ArrDelayMinutes` was used to derive the label and then removed, so it cannot accidentally enter a future feature pipeline.

## 5. Chronological train, validation, and test splits

A random split would allow later flight patterns to influence training for earlier observations. Chronological splitting better represents the real task: learn from the past and predict a later period.

**Locked split contract v1:** train = 2023-01-01 through 2024-04-30; validation = 2024-05-01 through 2024-08-31; test = 2024-09-01 through 2024-12-31. These calendar boundaries are fixed and must not be moved in response to model scores or target prevalence. The final four months are a previously inspected historical test period, not a pristine external holdout.

The observed delayed-flight target prevalence shifts from **26.1% in validation to 16.1% in test** (26.125% to 16.114%, a **-10.011 percentage-point** change). This is a material temporal distribution shift to report, not a reason to redraw the split or tune on the test period.

In [3]:
LOCKED_SPLIT_CONTRACT_V1 = {
    "train": ("2023-01-01", "2024-04-30"),
    "validation": ("2024-05-01", "2024-08-31"),
    "test": ("2024-09-01", "2024-12-31"),
}

train = eligible.filter(F.col("flight_date").between(*LOCKED_SPLIT_CONTRACT_V1["train"]))
validation = eligible.filter(F.col("flight_date").between(*LOCKED_SPLIT_CONTRACT_V1["validation"]))
test = eligible.filter(F.col("flight_date").between(*LOCKED_SPLIT_CONTRACT_V1["test"]))

def split_summary(frame, split_name):
    return (
        frame.agg(
            F.count("*").alias("rows"),
            F.sum("label").alias("delayed_rows"),
            F.round(F.avg("label") * 100, 3).alias("delayed_proportion_pct"),
            F.min("flight_date").alias("first_date"),
            F.max("flight_date").alias("last_date"),
        )
        .withColumn("split", F.lit(split_name))
        .select("split", "rows", "delayed_rows", "delayed_proportion_pct", "first_date", "last_date")
    )

split_statistics = (
    split_summary(train, "train")
    .unionByName(split_summary(validation, "validation"))
    .unionByName(split_summary(test, "test"))
)
split_rows = {row["split"]: row for row in split_statistics.collect()}
spark.createDataFrame(list(split_rows.values()), split_statistics.schema).show(truncate=False)

validation_target_pct = float(split_rows["validation"]["delayed_proportion_pct"])
test_target_pct = float(split_rows["test"]["delayed_proportion_pct"])
target_shift_pp = test_target_pct - validation_target_pct
print(
    f"Observed target prevalence shift: validation {validation_target_pct:.1f}% -> "
    f"test {test_target_pct:.1f}% ({target_shift_pp:+.3f} percentage points)."
)

+----------+-------+------------+----------------------+----------+----------+
|split     |rows   |delayed_rows|delayed_proportion_pct|first_date|last_date |
+----------+-------+------------+----------------------+----------+----------+
|train     |8946370|1827125     |20.423                |2023-01-01|2024-04-30|
|validation|2418739|631898      |26.125                |2024-05-01|2024-08-31|
|test      |2343541|377642      |16.114                |2024-09-01|2024-12-31|
+----------+-------+------------+----------------------+----------+----------+

Observed target prevalence shift: validation 26.1% -> test 16.1% (-10.011 percentage points).


### Split integrity checks

The following assertions verify that every eligible row falls in exactly one v1 date window, that observed date bounds match the locked contract, and that the documented 26.1% to 16.1% target shift remains visible. A changed assertion requires an explicit data/version review—not an automatic boundary adjustment.

In [4]:
eligible_count = eligible.count()
train_count = train.count()
validation_count = validation.count()
test_count = test.count()
outside_split_windows = eligible.filter(
    (F.col("flight_date") < F.lit(LOCKED_SPLIT_CONTRACT_V1["train"][0]).cast("date"))
    | (F.col("flight_date") > F.lit(LOCKED_SPLIT_CONTRACT_V1["test"][1]).cast("date"))
).count()

assert outside_split_windows == 0, "Eligible rows exist outside the defined date horizon."
assert train_count + validation_count + test_count == eligible_count, "Split counts do not cover the eligible population exactly once."
assert train.filter(F.col("flight_date") > LOCKED_SPLIT_CONTRACT_V1["train"][1]).limit(1).count() == 0
assert validation.filter(
    (F.col("flight_date") < LOCKED_SPLIT_CONTRACT_V1["validation"][0])
    | (F.col("flight_date") > LOCKED_SPLIT_CONTRACT_V1["validation"][1])
).limit(1).count() == 0
assert test.filter(F.col("flight_date") < LOCKED_SPLIT_CONTRACT_V1["test"][0]).limit(1).count() == 0

observed_bounds = {
    name: (str(row["first_date"]), str(row["last_date"]))
    for name, row in split_rows.items()
}
assert observed_bounds == LOCKED_SPLIT_CONTRACT_V1, "Observed dates no longer match locked split contract v1."
assert round(validation_target_pct, 1) == 26.1
assert round(test_target_pct, 1) == 16.1

print("PASS: locked chronological split v1 is ordered, non-overlapping, complete, and unchanged.")

PASS: locked chronological split v1 is ordered, non-overlapping, complete, and unchanged.


## 6. Production implementation

Production fits the frozen preprocessing and Logistic Regression on train only, then explicitly evaluates validation and historical test with threshold 0.34. It does not tune during evaluation. The candidate fields above are exploratory; the canonical 50-feature allowlist is `airline.dws.ml_feature_contract.FEATURE_COLUMNS`.

Histories use rolling as-of-date semantics: earlier validation/test outcomes may contribute to later dates under daily availability. Histories never reset at split boundaries. October 20 uses September 20–October 19 for a 30-day history; same-day/future outcomes are excluded.


In [5]:
eligible.unpersist()
spark.stop()
print("Problem definition complete; temporary cache released and Spark session stopped.")

Problem definition complete; temporary cache released and Spark session stopped.
